# Tutorial: External Repositories
The interactive benckmark works based on the support of many existing tools. Breaking them down and taking a closer look at each function help you have a better understanding of how this tool works. This is especially useful when you encounter some bugs, because you know what went wrong.

## 1. CommonRoad Map Tool - crmapconverter

The repository [CommonRoad Map Tool](https://gitlab.lrz.de/cps/commonroad-map-tool)  provides multiple converters from different map formats to the CommonRoad map format. The function **convert_scenario_to_net_file** of the **crmapconverter** converts a CommonRoad scenario to the corresponding SUMO network.

In [5]:
import sys
sys.path.append("..")
import os
from configuration import CONFIG_TYPE, get_interactive_scenario_configuration
from configuration.config_1 import SumoConfig_1
import pickle
from crmapconverter.sumo_map.cr2sumo import CR2SumoMapConverter
from commonroad.scenario.scenario import ScenarioID
from commonroad.common.file_reader import CommonRoadFileReader

# Define the input and output path.
cr_scenario_path = "../example_scenarios/cr_scenario/USA_US101-26_2_T-1.xml"
config_type = CONFIG_TYPE.SUMO_CONFIG_1
output_folder_path = "../example_scenarios/output/tutorial_5/USA_US101-26_2_T-1-1"

# Define the configuration
benchmark_id = ScenarioID.from_benchmark_id(os.path.splitext(os.path.basename(cr_scenario_path))[0],
                                                scenario_version="2020a")
conf = get_interactive_scenario_configuration(config_type, str(benchmark_id))

# Load the CommonRoad scenario and create output folder
scenario, planning_problem_set = CommonRoadFileReader(cr_scenario_path).open()

os.makedirs(output_folder_path, exist_ok=True)

# Converter initialization
converter = CR2SumoMapConverter(scenario.lanelet_network, conf)
converter.scenario_name = conf.scenario_name

conversion_possible = converter.convert_scenario_to_net_file(scenario = scenario, 
                                                             output_folder = output_folder_path)

scenario_name = "USA_US101-26_2_T-1-1"
conf = SumoConfig_1.from_scenario_name(scenario_name)
conf.scenario_name = scenario_name
with open(os.path.join(output_folder_path, "simulation_config.p"), 'wb') as f:
    pickle.dump(conf, f)

We can see that in the given output folder there is a set of SUMO network created.

In [4]:
from os import listdir
from os.path import isfile, join

onlyfiles = [f for f in listdir(output_folder_path) if isfile(join(output_folder_path, f))]
print(onlyfiles)

['_connections.net.xml', 'USA_US101-26_2_T-1-1.add.xml', 'USA_US101-26_2_T-1-1.sumo.cfg', 'edges.net.xml', 'nodes.net.xml', '_tll.net.xml', 'simulation_config.p', 'USA_US101-26_2_T-1-1.pedestrians.rou.xml', 'USA_US101-26_2_T-1-1.vehicles.rou.xml', 'USA_US101-26_2_T-1-1.net.xml']


## 2. CommonRoad-SUMO interface - SumoSimulation, create_gif
The [CommonRoad-SUMO interface](https://gitlab.lrz.de/cps/sumo-interface) implements the interface between the framework CommonRoad and the traffic simulator SUMO. The interface is presented in detail in the [paper](https://mediatum.ub.tum.de/doc/1486856/344641.pdf) and a documentation of the API can be found [here](https://commonroad.in.tum.de/static/docs/commonroad-sumo-interface/index.html).

In our interactive benchmark, we take the use of its **SumoSimulation** for simulating scenarios in SUMO and use **create_gif** for generating gif files.


In [8]:
import pickle
import os

from sumocr.visualization.gif import create_gif
from sumocr.maps.scenario_wrapper import AbstractScenarioWrapper

from commonroad.common.file_reader import CommonRoadFileReader
from commonroad.common.solution import CommonRoadSolutionReader
import copy

# Define arguments.
interactive_scenario_folder = "../example_scenarios/output/tutorial_1/USA_US101-26_2_T-1-1"
solution_file = "../example_scenarios/solution/KS2:SM1:USA_US101-26_2_T-1:2020a.xml"
output_folder_path = "../example_scenarios/gif"
creating_video = True 
use_sumo_manager = False

# Load simulation configuration file.
with open(os.path.join(interactive_scenario_folder, "simulation_config.p"), "rb") as input_file:
    conf = pickle.load(input_file)
num_of_steps = conf.simulation_steps

# Load CommonRoad scenario on its initial state from the .cr.xml file.
scenario_file = os.path.join(interactive_scenario_folder, f"{conf.scenario_name}.cr.xml")
scenario, planning_problem_set = CommonRoadFileReader(scenario_file).open()

# Define the scenario warpper.
scenario_wrapper = AbstractScenarioWrapper()
scenario_wrapper.sumo_cfg_file = os.path.join(interactive_scenario_folder,
                                              f"{conf.scenario_name}.sumo.cfg")
scenario_wrapper.lanelet_network = scenario.lanelet_network

# Load the trajectory of the ego vehicle (solution).
solution = CommonRoadSolutionReader.open(solution_file)



In [9]:
from sumocr.interface.sumo_simulation import SumoSimulation

# INitialize SumoSImulation
sumo_sim = SumoSimulation()

sumo_sim.planning_problem_set = planning_problem_set
sumo_sim.initialize(conf, scenario_wrapper)

def run_simulation():
    for step in range(num_of_steps):
        # plan trajectories for all ego vehicles
        if solution is not None:
            ego_vehicles = sumo_sim.ego_vehicles
            commonroad_scenario = sumo_sim.commonroad_scenario_at_time_step(
                sumo_sim.current_time_step)

            for idx, ego_vehicle in enumerate(ego_vehicles.values()):
                current_state = ego_vehicle.current_state

                # Use the solution trajectory
                ego_trajectory = solution.planning_problem_solutions[idx].trajectory
                if len(ego_trajectory.state_list) > step:
                    next_state = copy.deepcopy(ego_trajectory.state_list[step])
                else:
                    return
                next_state.time_step = 1
                ego_trajectory: List[State] = [next_state]
                ego_vehicle.set_planned_trajectory(ego_trajectory)

            # Set the modified ego vehicles to synchronize in case of sumo-manager
            sumo_sim.ego_vehicles = ego_vehicles
        else:
            sumo_sim.dummy_ego_simulation = True

        sumo_sim.simulate_step()

# Run the simulation
run_simulation()

# Get the simulated scenario from the simulation interface.
simulated_scenario = sumo_sim.commonroad_scenarios_all_time_steps()
sumo_sim.stop()
simulated_scenario.scenario_id = scenario.scenario_id

 Retrying in 1 seconds


Similar to what we have done in Tutorial 2, an interactive sceranio is already simulated. Then we utilize the **create_gif** function to generate the gif and save it to the output folder.

In [10]:
for idx, planning_problem in enumerate(planning_problem_set.planning_problem_dict.values()):
    create_gif(simulated_scenario, output_folder_path,
               planning_problem=planning_problem,
               trajectory=solution.planning_problem_solutions[idx].trajectory,
               secondary_scenario=None,
               follow_ego=True)

Saving USA_US101-26_2_T-1.gif
USA_US101-26_2_T-1.gif saved


Please run the Markdown block to show the gif.
![SegmentLocal](../example_scenarios/gif/USA_US101-26_2_T-1.gif "segment")

## 3. SUMO manager - SumoRPCClient and Docker Image